<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week8/Day5/RAG_HuggingFace_LangChain_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [2]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 

In [1]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


/tmp/ipykernel_7831/1013419477.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [2]:
import requests
import datasets
import transformers
import langchain
import langchain_community
import langchain_huggingface
import faiss

print("Requests version:", requests.__version__)
print("All imports successful!")

Requests version: 2.34.2
All imports successful!


## 1) Load dataset and convert to Documents


In [3]:
from datasets import load_dataset

# Load the Hugging Face documentation dataset
dataset = load_dataset("m-ric/huggingface_doc")

# Display dataset structure
print(dataset)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'source'],
        num_rows: 2647
    })
})


In [4]:
# Inspect dataset columns
print("Columns:", dataset["train"].column_names)

# Display one example document
example = dataset["train"][0]

print("\nSource:", example["source"])
print("\nText preview:")
print(example["text"][:1000])

Columns: ['text', 'source']

Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx

Text preview:
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />

## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.

<img src="https://raw.git

In [5]:
from langchain_core.documents import Document

# Convert dataset rows into LangChain Documents
documents = [
    Document(
        page_content=row["text"],
        metadata={"source": row["source"]}
    )
    for row in dataset["train"]
]

print("Total documents:", len(documents))
print("First document source:", documents[0].metadata["source"])
print("First document preview:", documents[0].page_content[:200])

Total documents: 2647
First document source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
First document preview:  Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deplo


## 2) Split into chunks


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# First chunking configuration
text_splitter_1 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_1 = text_splitter_1.split_documents(documents)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks_1))
print("First chunk length:", len(chunks_1[0].page_content))
print("\nFirst chunk preview:")
print(chunks_1[0].page_content)

Original documents: 2647
Total chunks: 61227
First chunk length: 424

First chunk preview:
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:


In [7]:
# Second chunking configuration
text_splitter_2 = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks_2 = text_splitter_2.split_documents(documents)

print("Configuration 1 (500/50):", len(chunks_1), "chunks")
print("Configuration 2 (1000/100):", len(chunks_2), "chunks")

print("\nFirst chunk from configuration 2:")
print(chunks_2[0].page_content)

Configuration 1 (500/50): 61227 chunks
Configuration 2 (1000/100): 28297 chunks

First chunk from configuration 2:
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />

## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.

<img src="https://raw.githubusercontent.com/hugging

## 3) Vector store + retriever (FAISS)


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embedding dimensions
test_vector = embeddings.embed_query(
    "How do I create a Hugging Face endpoint?"
)

print("Embedding dimensions:", len(test_vector))
print("Embedding model loaded successfully!")

Embedding dimensions: 384
Embedding model loaded successfully!


In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 64}
)

In [10]:
from langchain_community.vectorstores import FAISS
import time

# Measure indexing time
start_time = time.time()

# Generate embeddings on GPU and build FAISS index
vector_store = FAISS.from_documents(
    documents=chunks_2,
    embedding=embeddings
)

elapsed_time = time.time() - start_time

print("FAISS vector store created successfully!")
print("Total indexed chunks:", vector_store.index.ntotal)
print(f"Processing time: {elapsed_time:.2f} seconds")

FAISS vector store created successfully!
Total indexed chunks: 28297
Processing time: 78.83 seconds


In [11]:
# Create a retriever with top k=4
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever created successfully!")
print("Top K:", retriever.search_kwargs["k"])

Retriever created successfully!
Top K: 4


In [12]:
# Test document retrieval
question = "How can I create a Hugging Face Inference Endpoint?"

retrieved_docs = retriever.invoke(question)

print("Question:", question)
print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Document {i} ---")
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Content:", doc.page_content[:500])

Question: How can I create a Hugging Face Inference Endpoint?
Retrieved documents: 4

--- Document 1 ---
Source: huggingface/blog/blob/main/mantis-case-study.md
Content: _If you’re interested in Hugging Face Inference Endpoints for your company, please contact us [here](https://huggingface.co/inference-endpoints/enterprise) - our team will contact you to discuss your requirements!_

_This article was originally published on February 15, 2023 [in Medium](https://medium.com/mantisnlp/why-were-switching-to-hugging-face-inference-endpoints-and-maybe-you-should-too-829371dcd330)._

--- Document 2 ---
Source: huggingface/huggingface_hub/blob/main/docs/source/en/guides/inference.md
Content: </Tip>

## Supported tasks

[`InferenceClient`]'s goal is to provide the easiest interface to run inference on Hugging Face models. It
has a simple API that supports the most common tasks. Here is a list of the currently supported tasks:

--- Document 3 ---
Source: huggingface/blog/blob/main/inference-upda

In [13]:
# Compare retrieval results with different k values
for k in [2, 6]:
    test_retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )

    results = test_retriever.invoke(question)

    print(f"\n{'=' * 60}")
    print(f"RETRIEVAL WITH K={k}")
    print("=" * 60)

    for i, doc in enumerate(results, start=1):
        print(f"\nRank {i}")
        print("Source:", doc.metadata.get("source", "Unknown"))
        print("Content:", doc.page_content[:300])


RETRIEVAL WITH K=2

Rank 1
Source: huggingface/blog/blob/main/mantis-case-study.md
Content: _If you’re interested in Hugging Face Inference Endpoints for your company, please contact us [here](https://huggingface.co/inference-endpoints/enterprise) - our team will contact you to discuss your requirements!_

_This article was originally published on February 15, 2023 [in Medium](https://medi

Rank 2
Source: huggingface/huggingface_hub/blob/main/docs/source/en/guides/inference.md
Content: </Tip>

## Supported tasks

[`InferenceClient`]'s goal is to provide the easiest interface to run inference on Hugging Face models. It
has a simple API that supports the most common tasks. Here is a list of the currently supported tasks:

RETRIEVAL WITH K=6

Rank 1
Source: huggingface/blog/blob/main/mantis-case-study.md
Content: _If you’re interested in Hugging Face Inference Endpoints for your company, please contact us [here](https://huggingface.co/inference-endpoints/enterprise) - our team will conta

## 4) Build the RAG chain


In [14]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# Load FLAN-T5 on GPU
generator = pipeline(
    task="text2text-generation",
    model="google/flan-t5-small",
    device=0,
    max_new_tokens=150
)

# Wrap the model for LangChain
llm = HuggingFacePipeline(pipeline=generator)

print("FLAN-T5 model loaded successfully!")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


FLAN-T5 model loaded successfully!


In [16]:
%pip install -q langchain-classic

In [17]:
from langchain_classic.chains import RetrievalQA

print("RetrievalQA imported successfully!")

RetrievalQA imported successfully!


In [18]:
# Build the RAG question-answering chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("RAG pipeline created successfully!")

RAG pipeline created successfully!


## 5) Demo: RAG vs no-RAG


In [20]:
# Test RAG with a more specific question
question = (
    "Create an Endpoint. Enter the Hugging Face Repository ID "
    "and endpoint name. Select Cloud Provider and region."
)

response = qa_chain.invoke({"query": question})

print("Question:", question)
print("\nAnswer:", response["result"])

print("\nSources:")
for i, doc in enumerate(response["source_documents"], start=1):
    print(f"{i}. {doc.metadata.get('source', 'Unknown')}")

Token indices sequence length is longer than the specified maximum sequence length for this model (933 > 512). Running this sequence through the model will result in indexing errors


Question: Create an Endpoint. Enter the Hugging Face Repository ID and endpoint name. Select Cloud Provider and region.

Answer: Create an Endpoint.

Sources:
1. huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
2. huggingface/transformers/blob/main/examples/flax/README.md
3. huggingface/blog/blob/main/hugging-face-endpoints-on-azure.md
4. huggingface/blog/blob/main/hugging-face-endpoints-on-azure.md


In [21]:
from langchain_core.documents import Document

# Retrieve only the two most relevant documents
short_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# Retrieve and shorten the context
question = (
    "Create an Endpoint. Enter the Hugging Face Repository ID "
    "and endpoint name. Select Cloud Provider and region."
)

short_docs = short_retriever.invoke(question)

context = "\n\n".join(
    doc.page_content[:500] for doc in short_docs
)

prompt = (
    "Answer the question using only the context below.\n\n"
    f"Context:\n{context}\n\n"
    "Question: What are the first steps to create a "
    "Hugging Face Inference Endpoint?\n"
    "Answer:"
)

# Check input length before generation
token_count = len(generator.tokenizer(prompt)["input_ids"])

print("Input tokens:", token_count)
print("Within model limit:", token_count <= 512)

Input tokens: 325
Within model limit: True


In [22]:
# Generate an answer using the shortened context
answer = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print("Question: What are the first steps to create a Hugging Face Inference Endpoint?")
print("\nAnswer:", answer[0]["generated_text"])

print("\nSources:")
for i, doc in enumerate(short_docs, start=1):
    print(f"{i}. {doc.metadata.get('source', 'Unknown')}")

Question: What are the first steps to create a Hugging Face Inference Endpoint?

Answer: you will need to be logged in to the Hugging Face website locally for it to work, the easiest way to achieve this is to run huggingface-cli login and then type your username and password when prompted. You can also pass along your authentication token with the --hub_token argument. - the output_dir you pick will either need to be a new folder or a local clone

Sources:
1. huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
2. huggingface/transformers/blob/main/examples/flax/README.md


In [23]:
# Select the official endpoint creation guide
official_doc = next(
    doc for doc in short_docs
    if "create_endpoint.mdx" in doc.metadata["source"]
)

# Build a focused prompt
prompt = (
    "Based only on the documentation below, "
    "list the first two steps to create an Inference Endpoint.\n\n"
    f"Documentation:\n{official_doc.page_content[:650]}\n\n"
    "Answer:"
)

# Check token count and generate the answer
token_count = len(generator.tokenizer(prompt)["input_ids"])
print("Input tokens:", token_count)

if token_count <= 512:
    answer = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )
    print("\nAnswer:", answer[0]["generated_text"])

print("\nSource:", official_doc.metadata["source"])

Input tokens: 244

Answer: Open the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub app. Select the GitHub

Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


In [24]:
# Test FLAN-T5 with a simple, explicit context
test_prompt = (
    "Context: To create a Hugging Face Inference Endpoint, "
    "first enter the Repository ID and endpoint name. "
    "Next, select the Cloud Provider and region.\n\n"
    "Question: What are the first two steps?\n"
    "Answer:"
)

test_result = generator(
    test_prompt,
    max_new_tokens=60,
    do_sample=False,
    repetition_penalty=1.5,
    no_repeat_ngram_size=3
)

print("Answer:", test_result[0]["generated_text"])

Answer: Enter the Repository ID and endpoint name.


In [25]:
# Extract the relevant section from the retrieved document
context = official_doc.page_content[:650]

prompt = (
    "Context:\n"
    f"{context}\n\n"
    "Question: What information must the user enter first "
    "when creating a Hugging Face Inference Endpoint?\n"
    "Answer:"
)

result = generator(
    prompt,
    max_new_tokens=60,
    do_sample=False,
    repetition_penalty=1.5,
    no_repeat_ngram_size=3
)

print("Answer:", result[0]["generated_text"])
print("Source:", official_doc.metadata["source"])

Answer: Hugging Face Repository ID
Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


In [26]:
# Ask for both required fields using the official documentation
prompt = (
    "Context:\n"
    f"{official_doc.page_content[:650]}\n\n"
    "Question: Which two fields must be entered in step 1 "
    "of creating a Hugging Face Inference Endpoint?\n"
    "Answer with both field names:"
)

result = generator(
    prompt,
    max_new_tokens=60,
    do_sample=False,
    repetition_penalty=1.5,
    no_repeat_ngram_size=3
)

print("Answer:", result[0]["generated_text"])
print("Source:", official_doc.metadata["source"])

Answer: [Endpoint creation page]
Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


In [27]:
import re

# Extract the relevant instructions from the official document
text = official_doc.page_content

step_1 = re.search(
    r"## 1\.\s*(.*?)(?=\n|$)",
    text
)

step_2 = re.search(
    r"## 2\.\s*(.*?)(?=\n|$)",
    text
)

clean_context = "\n".join(
    match.group(1).strip()
    for match in [step_1, step_2]
    if match
)

print("Clean context:")
print(clean_context)

Clean context:
Enter the Hugging Face Repository ID and your desired endpoint name:
Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.


In [28]:
# Generate an answer using clean retrieved context
prompt = (
    f"Context:\n{clean_context}\n\n"
    "Question: What are the first two steps to create "
    "a Hugging Face Inference Endpoint?\n"
    "Answer:"
)

result = generator(
    prompt,
    max_new_tokens=80,
    do_sample=False,
    repetition_penalty=1.5,
    no_repeat_ngram_size=3
)

print("Answer:", result[0]["generated_text"])
print("Source:", official_doc.metadata["source"])

Answer: Select your Cloud Provider and region.
Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


In [29]:
# Test each instruction separately
questions = [
    "What information must be entered in step 1?",
    "What must be selected in step 2?"
]

for question in questions:
    prompt = (
        f"Context:\n{clean_context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    result = generator(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3
    )

    print("\nQuestion:", question)
    print("Answer:", result[0]["generated_text"])

print("\nSource:", official_doc.metadata["source"])


Question: What information must be entered in step 1?
Answer: Hugging Face Repository ID

Question: What must be selected in step 2?
Answer: Cloud Provider and region

Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


### Task 7 — RAG Evaluation

The RAG pipeline was implemented using LangChain, FAISS, and the local `google/flan-t5-small` model.

The retrieval process successfully identified the official Hugging Face Inference Endpoints documentation.

Initial answers were incomplete because the retrieved context exceeded the model's 512-token input limit and contained irrelevant information.

After reducing and cleaning the context, the model successfully answered individual questions about the endpoint creation process.

**Conclusion:** Retrieval quality, context length, and prompt design significantly affect RAG performance. Smaller language models may require shorter contexts and more focused questions to produce reliable answers.